In [0]:
def write_user_post_geolocation_table(spark, run_date, db, table_names): 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_post_geolocation} PARTITION (partition_date = '{run_date}')
              with user_provinces as (
                select customer_id as user_id, 
                last(case when ip_address RLIKE '[A-Za-z0-9]' then 'overseas'
                else regexp_replace(ip_address, '[省市]', '')
                end) as province
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer
                where coalesce(to_date(from_unixtime(update_at / 1000)),to_date(from_unixtime(create_at / 1000))) <= '{run_date}'
                and to_date(from_unixtime(register_time / 1000)) <= '{run_date}'
                and ip_address is not null
                and delete_type = 0
                and `identity` != 2
                and agree_community_time is not null
                group by 1
              ),
              posts_published as (
                select distinct customer_id as user_id, 
                `id` as post_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where to_date(from_unixtime(post_time/1000)) = '{run_date}'
                and audit_status != 2 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and is_hide = 0  
                and publish_approval_status not in (0, 4) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
              ),
              post_provinces as (
                select pp.user_id, pp.post_id, up.province
                from posts_published pp
                inner join user_provinces up
                on pp.user_id = up.user_id
              )
              select up.user_id,
              pp.post_id,
              pp.province
              from user_provinces up
              inner join post_provinces pp
              on up.province = pp.province
              where up.user_id != pp.user_id --exclude posts published by the authors themselves
              """.format(db = db,
                         user_post_geolocation = table_names["user_post_geolocation"],
                         run_date = run_date)
    )

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_user_post_geolocation_table(spark, run_date, db, table_names)